## Visualization and Storytelling

### Introduction

This analysis provides a data-driven overview of the current technology job market based on real
LinkedIn job postings data. The objective is to support hiring and workforce planning decisions by
identifying patterns in job demand, required skills, work modalities, and compensation.

The target audience is hiring managers and HR decision-makers who need clear, actionable insights
to better understand market trends and align recruitment strategies with current demand.


### Data sources and preparation

The analysis is based on multiple datasets extracted and processed from LinkedIn job postings,
including company information, job postings, required skills, industries, employee size, and salary data.


In [34]:
import pandas as pd

companies = pd.read_csv("../data/processed/companies_clean.csv")
postings = pd.read_csv("../data/processed/postings_clean.csv")
salaries = pd.read_csv("../data/processed/salaries_clean.csv")

job_skills = pd.read_csv("../data/raw/linkedin-job-postings/jobs/job_skills.csv")
skills = pd.read_csv("../data/raw/linkedin-job-postings/mappings/skills.csv")

companies_industries = pd.read_csv("../data/raw/linkedin-job-postings/companies/company_industries.csv")
job_industries = pd.read_csv("../data/raw/linkedin-job-postings/jobs/job_industries.csv")
industries = pd.read_csv("../data/raw/linkedin-job-postings/mappings/industries.csv")

employee_counts = pd.read_csv("../data/raw/linkedin-job-postings/companies/employee_counts.csv")


## Which job roles are most in demand?

To understand which roles are most in demand, job demand is measured as the number of job postings
per job title. Although job titles are not fully standardized across companies, they provide a
practical and widely used proxy for identifying job roles at scale.

This analysis highlights the roles that appear most frequently in the job market, offering insight
into where hiring activity is currently concentrated.

The following ranking shows the top job titles by number of postings.

In [35]:
job_demand_by_role = (
    postings
    .groupby('title')
    .size()
    .reset_index(name='num_postings')
    .sort_values('num_postings', ascending=False)
)

job_demand_by_role.head(10)

,title,num_postings
54901,Sales Manager,673
15069,Customer Service Representative,373
47243,Project Manager,354
2510,Administrative Assistant,254
56343,Senior Accountant,238
21744,Executive Assistant,228
55357,Salesperson,211
50465,Registered Nurse,210
49827,Receptionist,204
64091,Staff Accountant,200


## What skills appear most frequently across job postings?

To identify the most in-demand skills, skill demand is measured by counting how often each skill
appears across job postings. This reflects the popularity of skills in the market rather than
proficiency level or seniority requirements.

Understanding which skills are most frequently requested helps hiring managers align job profiles,
training plans, and talent acquisition strategies with current market needs.

In [36]:
skill_demand = (
    job_skills
    .merge(skills[['skill_abr', 'skill_name']], on='skill_abr', how='left')
    .dropna(subset=['skill_name'])
    .groupby('skill_name')
    .size()
    .reset_index(name='num_postings')
    .sort_values('num_postings', ascending=False)
)

skill_demand.head(10)

,skill_name,num_postings
16,Information Technology,26137
29,Sales,22475
18,Management,20861
19,Manufacturing,18185
14,Health Care Provider,17369
5,Business Development,14290
11,Engineering,13009
21,Other,12608
12,Finance,8540
20,Marketing,5525


## How does job demand vary by location and industry?

This section shows how job demand varies across geographies and industries.
Location demand is measured using the `location` field from job postings.

Industry demand is approximated by assigning each posting the industry of the hiring company
(company-level classification). This provides a consistent view of demand by sector and avoids
possible duplication from multi-label job-level industry tags.

In [37]:
postings_with_industry = postings.merge(companies_industries, on="company_id", how="left")

job_demand_by_industry = (
    postings_with_industry
    .groupby("industry")["job_id"]
    .nunique()
    .reset_index(name="num_postings")
    .sort_values("num_postings", ascending=False)
)

job_demand_by_industry.head(10)

,industry,num_postings
123,Staffing and Recruiting,18886
54,Hospitals and Health Care,15753
56,IT Services and IT Consulting,11573
112,Retail,9642
120,Software Development,5729
40,Financial Services,5516
27,Construction,1969
53,Hospitality,1913
86,Non-profit Organizations,1904
106,Real Estate,1868


In [38]:
job_demand_by_location = (
    postings
    .groupby("location")["job_id"]
    .nunique()
    .reset_index(name="num_postings")
    .sort_values("num_postings", ascending=False)
)

job_demand_by_location.head(10)

,location,num_postings
7733,United States,8125
5412,"New York, NY",2756
1359,"Chicago, IL",1834
3576,"Houston, TX",1762
1819,"Dallas, TX",1383
298,"Atlanta, GA",1363
763,"Boston, MA",1176
348,"Austin, TX",1083
1296,"Charlotte, NC",1075
6039,"Phoenix, AZ",1059


In [39]:
postings_with_industry = postings.merge(
    companies_industries,
    on="company_id",
    how="left"
)

top_industries = (
    postings_with_industry
    .groupby("industry")["job_id"].nunique()
    .sort_values(ascending=False)
    .head(10)
    .index
)

top_locations = (
    postings_with_industry
    .groupby("location")["job_id"].nunique()
    .sort_values(ascending=False)
    .head(10)
    .index
)

loc_ind_matrix = (
    postings_with_industry[
        postings_with_industry["industry"].isin(top_industries)
        & postings_with_industry["location"].isin(top_locations)
    ]
    .groupby(["location", "industry"])["job_id"].nunique()
    .reset_index(name="num_postings")
)

loc_ind_matrix.head(20)


,location,industry,num_postings
0,"Atlanta, GA",Construction,21
1,"Atlanta, GA",Financial Services,102
2,"Atlanta, GA",Hospitality,32
3,"Atlanta, GA",Hospitals and Health Care,85
4,"Atlanta, GA",IT Services and IT Consulting,184
5,"Atlanta, GA",Non-profit Organizations,14
6,"Atlanta, GA",Real Estate,42
7,"Atlanta, GA",Retail,44
8,"Atlanta, GA",Software Development,111
9,"Atlanta, GA",Staffing and Recruiting,227


In [40]:
loc_ind_pivot = (
    loc_ind_matrix
    .pivot(index="location", columns="industry", values="num_postings")
    .fillna(0)
    .astype(int)
)

loc_ind_pivot

industry,Construction,Financial Services,Hospitality,Hospitals and Health Care,IT Services and IT Consulting,Non-profit Organizations,Real Estate,Retail,Software Development,Staffing and Recruiting
location,,,,,,,,,,
"Atlanta, GA",21,102,32,85,184,14,42,44,111,227
"Austin, TX",17,39,30,55,203,6,38,73,120,127
"Boston, MA",11,70,17,151,109,16,6,22,79,249
"Charlotte, NC",16,116,8,94,183,9,19,74,29,165
"Chicago, IL",13,155,45,135,234,69,41,57,113,289
"Dallas, TX",29,89,34,141,217,9,34,43,76,194
"Houston, TX",35,66,26,126,154,18,44,81,50,333
"New York, NY",13,388,40,162,214,70,52,56,195,602
"Phoenix, AZ",34,69,19,102,107,24,25,56,41,156


In [41]:
col_order = loc_ind_pivot.sum(axis=0).sort_values(ascending=False).index
row_order = loc_ind_pivot.sum(axis=1).sort_values(ascending=False).index

loc_ind_pivot_sorted = loc_ind_pivot.loc[row_order, col_order]
loc_ind_pivot_sorted

industry,Staffing and Recruiting,IT Services and IT Consulting,Software Development,Hospitals and Health Care,Financial Services,Retail,Real Estate,Non-profit Organizations,Hospitality,Construction
location,,,,,,,,,,
United States,1009,1728,1534,443,284,60,43,64,25,27
"New York, NY",602,214,195,162,388,56,52,70,40,13
"Chicago, IL",289,234,113,135,155,57,41,69,45,13
"Houston, TX",333,154,50,126,66,81,44,18,26,35
"Dallas, TX",194,217,76,141,89,43,34,9,34,29
"Atlanta, GA",227,184,111,85,102,44,42,14,32,21
"Boston, MA",249,109,79,151,70,22,6,16,17,11
"Charlotte, NC",165,183,29,94,116,74,19,9,8,16
"Austin, TX",127,203,120,55,39,73,38,6,30,17


## What proportion of roles are remote or on-site?

Remote job availability is analyzed using the `remote_allowed` flag, which identifies
job postings that explicitly allow remote work. This analysis focuses on the proportion
of roles with declared remote availability, rather than a full remote / hybrid / on-site
classification.

It is important to note that missing values in this field do not necessarily indicate
on-site roles, but rather the absence of explicit remote information in the job posting.

In [42]:
remote_overview = (
    postings
    .assign(
        remote_flag=postings["remote_allowed"].fillna(0)
    )
    .groupby("remote_flag")["job_id"]
    .nunique()
    .reset_index(name="num_postings")
)

remote_overview["remote_flag"] = remote_overview["remote_flag"].map({
    1: "Remote allowed",
    0: "Not explicitly remote"
})

remote_overview


,remote_flag,num_postings
0,Not explicitly remote,108603
1,Remote allowed,15246


In [43]:
total_postings = remote_overview["num_postings"].sum()

remote_overview["share"] = (
    remote_overview["num_postings"] / total_postings
)

remote_overview


,remote_flag,num_postings,share
0,Not explicitly remote,108603,0.876898
1,Remote allowed,15246,0.123102


In [44]:
remote_postings = postings[postings["remote_allowed"] == 1]

remote_with_industry = remote_postings.merge(
    companies_industries,
    on="company_id",
    how="left"
)

remote_by_industry = (
    remote_with_industry
    .groupby("industry")["job_id"]
    .nunique()
    .reset_index(name="num_remote_postings")
)

total_by_industry = (
    postings
    .merge(companies_industries, on="company_id", how="left")
    .groupby("industry")["job_id"]
    .nunique()
    .reset_index(name="total_postings")
)

remote_share_by_industry = (
    remote_by_industry
    .merge(total_by_industry, on="industry")
)

remote_share_by_industry["remote_ratio"] = (
    remote_share_by_industry["num_remote_postings"]
    / remote_share_by_industry["total_postings"]
)

remote_share_by_industry.sort_values("remote_ratio", ascending=False).head(10)


,industry,num_remote_postings,total_postings,remote_ratio
110,Strategic Management Services,1,1,1.000000
126,Wireless Services,1,1,1.000000
71,Mobile Gaming Apps,1,1,1.000000
27,E-Learning Providers,132,157,0.840764
116,Translation and Localization,52,66,0.787879
65,Market Research,37,48,0.770833
21,Computer and Network Security,176,305,0.577049
89,Professional Training and Coaching,52,91,0.571429
84,Photography,4,7,0.571429
20,Computer Networking Products,5,9,0.555556


This subsection highlights companies with a higher proportion of explicitly remote
job postings relative to their total hiring activity.


## How do salary levels vary across job roles?

To analyze how compensation varies across job roles, salary information was aggregated
at the job title level using annualized salary values. Median salary is used as the
primary indicator, as it is more robust to outliers than the mean.

To ensure meaningful comparisons, only roles with at least 10 job postings containing
salary information are included in the analysis.


In [45]:
# Merge salary data with job titles (roles) and keep annualized salary values
salary_postings = (
    salaries
    .merge(
        postings[["job_id", "title", "location"]],
        on="job_id",
        how="left"
    )
    .dropna(subset=["salary_yearly"])
)

# Aggregate salary by role (job title)
salary_by_role = (
    salary_postings
    .groupby("title")["salary_yearly"]
    .agg(
        median_salary="median",
        mean_salary="mean",
        num_postings="count"
    )
    .reset_index()
)

# Keep roles with enough salary data for meaningful comparison
salary_by_role = (
    salary_by_role[salary_by_role["num_postings"] >= 10]
    .sort_values("median_salary", ascending=False)
)

# (Optional) Top N roles for reporting / visualization
top_salary_roles = salary_by_role.head(15)

top_salary_roles


,title,median_salary,mean_salary,num_postings
3594,Chief Financial Officer,200000.0,218157.894737,19
15207,Primary Care Physician,200000.0,189000.000000,10
17494,Remote Licensed Mental Health Counselor,187200.0,188334.545455,11
9368,Head of Sales,182500.0,172800.000000,10
20316,Senior Site Reliability Engineer,155900.0,149080.000000,10
15361,Principal Software Engineer,149312.5,144302.500000,10
10590,Java Developer,145600.0,141032.000000,10
11461,Licensed Mental Health Therapist,145600.0,143104.000000,10
20823,Site Reliability Engineer,140400.0,142471.680000,10
20082,Senior Product Manager,140000.0,146019.769231,13


## How does location influence salary levels for similar roles?

In [46]:
# Base dataset
salary_postings = (
    salaries
    .merge(postings[["job_id", "title", "location"]], on="job_id", how="left")
    .dropna(subset=["salary_yearly", "title", "location"])
)

MIN_ROLE_POSTINGS = 30
top_roles = (
    salary_postings
    .groupby("title")["job_id"].nunique()
    .sort_values(ascending=False)
    .loc[lambda s: s >= MIN_ROLE_POSTINGS]
    .head(5)
    .index
)

salary_filtered = salary_postings[salary_postings["title"].isin(top_roles)]

salary_by_role_location = (
    salary_filtered
    .groupby(["title", "location"])["salary_yearly"]
    .agg(
        median_salary="median",
        mean_salary="mean",
        num_postings="count"
    )
    .reset_index()
)

MIN_LOC_POSTINGS_PER_ROLE = 3

salary_by_role_location = salary_by_role_location[
    salary_by_role_location["num_postings"] >= MIN_LOC_POSTINGS_PER_ROLE
]

top_locations_per_role = (
    salary_by_role_location
    .sort_values(["title", "num_postings"], ascending=[True, False])
    .groupby("title")
    .head(10)
)

top_locations_per_role = top_locations_per_role.sort_values(
    ["title", "median_salary"], ascending=[True, False]
)

top_locations_per_role.head(30)


,title,location,median_salary,mean_salary,num_postings
18,Administrative Assistant,"Dallas, TX",75000.0,68306.666667,3
54,Administrative Assistant,"New York, NY",66539.2,72251.840000,5
12,Administrative Assistant,"Chicago, IL",54760.0,44897.500000,4
86,Administrative Assistant,United States,52000.0,75504.000000,5
40,Administrative Assistant,"Jersey City, NJ",45760.0,55786.666667,3
74,Administrative Assistant,"San Diego, CA",37440.0,55896.533333,3
38,Administrative Assistant,"Honolulu, HI",35360.0,36053.333333,3
168,Customer Service Representative,United States,33280.0,29605.333333,3
190,Project Manager,"Charlotte, NC",143520.0,131040.000000,3
230,Project Manager,"New York, NY",110000.0,114184.685714,7
